<div style="background-color: #161b22; padding: 20px; border-radius: 12px; border-left: 6px solid #2ecc71; box-shadow: 0 4px 6px rgba(0,0,0,0.3);">
  <h1 style="color: #ffffff; margin: 0; font-family: sans-serif; font-weight: 700; letter-spacing: 1px;">
    ⚡ Z-Image Turbo <span style="font-size: 0.6em; color: #2ecc71; vertical-align: middle; background: #2ecc7122; padding: 2px 8px; border-radius: 6px;">Web UI · 6 LoRA Slots</span>
  </h1>
  <p style="color: #8b949e; margin: 8px 0 0 0; font-family: sans-serif;">
    • Hybrid FP8/GGUF + LoRA Optimized
  </p>
</div>

<p style="font-family: sans-serif; font-weight: 700; margin-top: 14px;"><svg width="18" height="18" viewBox="0 0 24 24" fill="#D97757" style="vertical-align:middle;margin-right:6px;" xmlns="http://www.w3.org/2000/svg"><path d="M12 2l2.2 6.1L20 6l-2.9 5.4L23 13l-6.3.7L18 20l-6-3.3L6 20l1.3-6.3L1 13l5.9-1.6L4 6l5.8 2.1L12 2z"/></svg>Brought to you by <a href="https://claude.com" target="_blank" style="color:#D97757; text-decoration:none;">Fable 5</a> <span style="font-weight:400;">🤖</span></p>

In [ ]:
#@title 1. Initialize Core Environment
#@markdown This prepares the ephemeral storage, installs ComfyUI, and configures the GGUF integration tools.

import os
import subprocess

LOCAL_WORKSPACE = "/content/ComfyUI"

print("🚀 Initializing Core Architecture...")
if not os.path.exists(LOCAL_WORKSPACE):
    !git clone https://github.com/comfyanonymous/ComfyUI {LOCAL_WORKSPACE} &> /dev/null
    print("   ✓ Core Engine Cloned")
else:
    !cd {LOCAL_WORKSPACE} && git pull &> /dev/null
    print("   ✓ Core Engine Updated")

print("📦 Installing Dependencies (This takes a moment)...")
!cd {LOCAL_WORKSPACE} && pip install xformers!=0.0.18 -r requirements.txt --extra-index-url https://download.pytorch.org/whl/cu121 &> /dev/null

GGUF_NODE_DIR = os.path.join(LOCAL_WORKSPACE, "custom_nodes/ComfyUI-GGUF")
if not os.path.exists(GGUF_NODE_DIR):
    print("🧩 Installing GGUF Processing Nodes...")
    !git clone https://github.com/city96/ComfyUI-GGUF {GGUF_NODE_DIR} &> /dev/null
    !pip install -r {GGUF_NODE_DIR}/requirements.txt &> /dev/null

print("✅ Environment Ready!")

In [ ]:
#@title 2. High-Speed Asset Downloader
#@markdown Paste your HuggingFace/Civitai model links here. **You can paste multiple LoRA URLs separated by commas or new lines** — they will all be downloaded to the loras folder. The required Z-Image base assets (Qwen Text Encoder & VAE) are automatically fetched.

import os
import subprocess
import gdown
import urllib.parse

WORKSPACE = "/content/ComfyUI"

# --- Input Resources ---
UNET_URLS = "" #@param {type:"string"}
LORA_URLS = "" #@param {type:"string"}
#@markdown *Optional: needed for Civitai models that require login (401/403 errors)*
CIVITAI_API_TOKEN = "" #@param {type:"string"}

# Pre-configured required models
TEXT_ENCODER_URLS = "https://huggingface.co/Comfy-Org/z_image_turbo/resolve/main/split_files/text_encoders/qwen_3_4b.safetensors"
VAE_URLS = "https://huggingface.co/Comfy-Org/z_image_turbo/resolve/main/split_files/vae/ae.safetensors"

DIRS = {
    "unet":           os.path.join(WORKSPACE, "models/unet"),
    "clip":           os.path.join(WORKSPACE, "models/clip"),
    "vae":            os.path.join(WORKSPACE, "models/vae"),
    "loras":          os.path.join(WORKSPACE, "models/loras"),
}

print("⚡ Configuring Aria2c Accelerator...")
subprocess.run(['apt-get', '-y', 'install', '-qq', 'aria2'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

def download_file(url, target_dir):
    try:
        os.makedirs(target_dir, exist_ok=True)
        before_files = set(os.listdir(target_dir))

        if "drive.google.com" in url:
            print(f"   📥 Downloading from Drive...")
            gdown.download(url, output=target_dir + '/', quiet=False, fuzzy=True)
        else:
            print(f"   📥 Fetching: {url.split('/')[-1][:40]}...")
            parsed_url = urllib.parse.urlparse(url)
            filename = os.path.basename(parsed_url.path)

            aria2_cmd = [
                "aria2c", "--console-log-level=error", "--summary-interval=10",
                "-c", "-x", "16", "-s", "16", "-k", "1M"
            ]

            is_civitai = "civitai" in parsed_url.netloc.lower()
            has_extension = os.path.splitext(filename)[1].lower() in (
                ".safetensors", ".gguf", ".ckpt", ".pt", ".pth", ".bin", ".sft", ".zip")

            if is_civitai and "/api/download/" in parsed_url.path and CIVITAI_API_TOKEN.strip():
                sep = "&" if parsed_url.query else "?"
                url = f"{url}{sep}token={CIVITAI_API_TOKEN.strip()}"

            if is_civitai or not has_extension:
                # Let the server name the file (follows redirects, reads Content-Disposition)
                aria2_cmd.extend(["--content-disposition", url, "-d", target_dir])
            else:
                aria2_cmd.extend(["-o", filename, url, "-d", target_dir])

            subprocess.run(aria2_cmd, check=True)

        after_files = set(os.listdir(target_dir))
        new_files = after_files - before_files

        if new_files:
            downloaded_file = list(new_files)[0]
            print(f"   ✅ Saved as: \033[96m{downloaded_file}\033[0m")
        else:
            print(f"   ⏭️ Already in library — skipped (existing file kept)")

    except Exception as e:
        print(f"   ❌ Failed: {url}\n      Error: {e}\n")

def process_downloads(urls_str, target_dir):
    if not urls_str.strip(): return
    url_list = [u.strip() for u in urls_str.replace(',', '\n').split('\n') if u.strip()]
    os.makedirs(target_dir, exist_ok=True)
    print(f"\n📂 Directory: {os.path.basename(target_dir)}")
    for url in url_list:
        download_file(url, target_dir)

process_downloads(UNET_URLS,         DIRS["unet"])
process_downloads(LORA_URLS,         DIRS["loras"])
process_downloads(TEXT_ENCODER_URLS, DIRS["clip"])
process_downloads(VAE_URLS,          DIRS["vae"])

print("\n✅ All assets secured!")

# --- Asset Library Inventory ---
def _human_size(num_bytes):
    for unit in ["B", "KB", "MB", "GB"]:
        if num_bytes < 1024 or unit == "GB":
            return f"{num_bytes:.1f} {unit}" if unit != "B" else f"{num_bytes} B"
        num_bytes /= 1024

import html as _htmlmod
import json as _jsonmod
from IPython.display import display as _display, HTML as _HTML

_inv = ['<div style="background:#161b22;border:1px solid #30363d;border-radius:10px;'
        'padding:14px 18px;font-family:monospace;margin:8px 0;max-width:640px;">'
        '<div style="color:#2ecc71;font-weight:bold;font-size:14px;">📋 ASSET LIBRARY</div>'
        '<div style="color:#8b949e;font-size:11px;margin-bottom:8px;">'
        'Click any name to copy it, then paste into the Generation cell</div>']
for label, key in [("🧠 UNet Models → UNET_FILENAME", "unet"),
                   ("🎨 LoRAs → LORA_1..6_FILENAME", "loras")]:
    folder = DIRS[key]
    entries = sorted(f for f in os.listdir(folder)) if os.path.exists(folder) else []
    entries = [f for f in entries if not f.endswith(".aria2")]
    _inv.append(f'<div style="color:#e6edf3;font-size:12px;margin:10px 0 4px 0;">{label}:</div>')
    if not entries:
        _inv.append('<div style="color:#8b949e;font-size:12px;margin-left:12px;">(empty)</div>')
    for f in entries:
        size = _human_size(os.path.getsize(os.path.join(folder, f)))
        safe_html = _htmlmod.escape(f)
        safe_js = _htmlmod.escape(_jsonmod.dumps(f), quote=True)
        _inv.append(
            f'<div style="margin:3px 0 3px 12px;">'
            f'<code onclick="navigator.clipboard.writeText({safe_js});'
            f'var b=this.nextElementSibling;b.textContent=\'✓ copied!\';'
            f'setTimeout(function(){{b.textContent=\'({size})\';}},1500);" '
            f'style="color:#58d6ff;background:#0d1117;border:1px solid #30363d;'
            f'border-radius:6px;padding:3px 8px;cursor:pointer;font-size:13px;" '
            f'title="Click to copy">{safe_html}</code> '
            f'<span style="color:#8b949e;font-size:11px;">({size})</span></div>')
_inv.append('</div>')
_display(_HTML("".join(_inv)))

In [ ]:
#@title 3. 🚀 Launch Web Studio
#@markdown Starts the ComfyUI engine and opens a **polished web UI** at a public link (gradio.live). All generation happens there — no more cell editing. Optional: set a password to keep the link private.

GRADIO_PASSWORD = "" #@param {type:"string"}

import sys
import os
import json
import time
import random
import subprocess
import urllib.request

WORKSPACE = "/content/ComfyUI"
COMFY_URL = "http://127.0.0.1:8188"
if os.path.isdir(WORKSPACE):
    os.chdir(WORKSPACE)

# ============================================================
# MODEL FAMILY REGISTRY (same verified registry as the form edition)
# ============================================================
FAMILIES = {
    "Z-Image Turbo": dict(
        loader="unet", clip=("CLIPLoader", "qwen_3_4b.safetensors", "lumina2"),
        vae="ae.safetensors", latent="EmptySD3LatentImage",
        shift=("ModelSamplingAuraFlow", 3.0), guidance=None, sampling="ksampler",
        defaults=dict(steps=9, cfg=1.0, sampler="res_multistep", scheduler="beta"),
        tip="Distilled turbo: keep CFG at 1.0. Sweet spot 9-14 steps. Shift 2.0-2.5 favors fine detail."),
    "FLUX.1 (dev/schnell/Krea)": dict(
        loader="unet", clip=("DualCLIPLoader", "clip_l.safetensors", "t5xxl_fp8_e4m3fn_scaled.safetensors", "flux"),
        vae="flux_ae.safetensors", latent="EmptySD3LatentImage",
        shift=None, guidance=3.5, sampling="ksampler",
        defaults=dict(steps=20, cfg=1.0, sampler="euler", scheduler="simple"),
        tip="dev/Krea-dev: 20-28 steps, guidance 3.5. schnell: only 4 steps! CFG always stays 1.0."),
    "FLUX.2 dev": dict(
        loader="unet", clip=("CLIPLoader", "mistral_3_small_flux2_fp8.safetensors", "flux2"),
        vae="flux2-vae.safetensors", latent="EmptyFlux2LatentImage",
        shift=None, guidance=4.0, sampling="flux2",
        defaults=dict(steps=20, cfg=1.0, sampler="euler", scheduler="simple"),
        tip="32B model — use a GGUF quant (Q4 or smaller) on Colab. Negative prompt is not used by this family."),
    "Chroma": dict(
        loader="unet", clip=("CLIPLoader", "t5xxl_fp8_e4m3fn_scaled.safetensors", "chroma"),
        vae="flux_ae.safetensors", latent="EmptySD3LatentImage",
        shift=("ModelSamplingAuraFlow", 1.0), guidance=None, sampling="ksampler",
        defaults=dict(steps=26, cfg=3.5, sampler="euler", scheduler="beta"),
        tip="Real CFG model: negative prompt works! Descriptive natural-language negatives work best."),
    "Qwen-Image": dict(
        loader="unet", clip=("CLIPLoader", "qwen_2.5_vl_7b_fp8_scaled.safetensors", "qwen_image"),
        vae="qwen_image_vae.safetensors", latent="EmptySD3LatentImage",
        shift=("ModelSamplingAuraFlow", 3.1), guidance=None, sampling="ksampler",
        defaults=dict(steps=20, cfg=2.5, sampler="euler", scheduler="simple"),
        tip="Excellent at text-in-image (incl. Chinese). With a lightning/distill LoRA: 8 steps, CFG 1.0."),
    "HiDream I1": dict(
        loader="unet", clip=("QuadrupleCLIPLoader", "clip_l_hidream.safetensors", "clip_g_hidream.safetensors", "t5xxl_fp8_e4m3fn_scaled.safetensors", "llama_3.1_8b_instruct_fp8_scaled.safetensors"),
        vae="flux_ae.safetensors", latent="EmptySD3LatentImage",
        shift=("ModelSamplingSD3", 6.0), guidance=None, sampling="ksampler",
        defaults=dict(steps=28, cfg=1.0, sampler="lcm", scheduler="normal"),
        tip="17B + 4 text encoders = VRAM heavy; use fp8/GGUF. dev: 28 steps CFG 1 | full: 50 steps CFG 5 | fast: 16 steps CFG 1."),
    "Anima": dict(
        loader="unet", clip=("CLIPLoader", "qwen_3_06b_base.safetensors", "stable_diffusion"),
        vae="qwen_image_vae.safetensors", latent="EmptyLatentImage",
        shift=None, guidance=None, sampling="ksampler",
        defaults=dict(steps=30, cfg=4.0, sampler="euler", scheduler="simple"),
        tip="Anime/illustration only (no realism). Danbooru tags AND natural language both work. With anima-turbo LoRA: 8 steps, CFG 1."),
    "SDXL": dict(
        loader="checkpoint", clip=None, vae=None, latent="EmptyLatentImage",
        shift=None, guidance=None, sampling="ksampler",
        defaults=dict(steps=28, cfg=6.0, sampler="dpmpp_2m", scheduler="karras"),
        tip="Native ~1024px. Real CFG model: negative prompt matters. LoRAs apply to model + text encoder."),
    "Pony": dict(
        loader="checkpoint", clip=None, vae=None, latent="EmptyLatentImage",
        shift=None, guidance=None, sampling="ksampler",
        defaults=dict(steps=25, cfg=7.0, sampler="euler_ancestral", scheduler="normal"),
        tip="Start prompts with: score_9, score_8_up, score_7_up. Source tags like source_anime help."),
    "Illustrious": dict(
        loader="checkpoint", clip=None, vae=None, latent="EmptyLatentImage",
        shift=None, guidance=None, sampling="ksampler",
        defaults=dict(steps=28, cfg=5.5, sampler="euler_ancestral", scheduler="normal"),
        tip="Danbooru-tag prompting (1girl, solo, ...). masterpiece/best quality tags help."),
    "NoobAI": dict(
        loader="checkpoint", clip=None, vae=None, latent="EmptyLatentImage",
        shift=None, guidance=None, sampling="ksampler",
        defaults=dict(steps=28, cfg=5.5, sampler="euler_ancestral", scheduler="normal"),
        tip="Illustrious-based, deep Danbooru/e621 tag knowledge. Artist tags are very strong."),
    "SD 1.5": dict(
        loader="checkpoint", clip=None, vae=None, latent="EmptyLatentImage",
        shift=None, guidance=None, sampling="ksampler",
        defaults=dict(steps=25, cfg=7.0, sampler="dpmpp_2m", scheduler="karras"),
        tip="Native 512px — use SD-tier resolutions or ~512-768 custom; going higher causes doubling artifacts."),
}
FAMILIES = {"Z-Image Turbo": FAMILIES["Z-Image Turbo"]}  # this edition: Z-Image only
DEFAULT_FAMILY = "Z-Image Turbo"

RESOLUTIONS = ["Custom", "1:1 · SD · 720x720", "1:1 · HD · 896x896", "1:1 · FHD · 1080x1080",
    "16:9 · SD · 1280x720", "16:9 · HD · 1600x896", "16:9 · FHD · 1920x1080",
    "9:16 · SD · 720x1280", "9:16 · HD · 896x1600", "9:16 · FHD · 1080x1920",
    "4:3 · SD · 960x720", "4:3 · HD · 1184x888", "4:3 · FHD · 1440x1080",
    "3:4 · SD · 720x960", "3:4 · HD · 888x1184", "3:4 · FHD · 1080x1440",
    "3:2 · SD · 1080x720", "3:2 · HD · 1344x896", "3:2 · FHD · 1632x1088",
    "2:3 · SD · 720x1080", "2:3 · HD · 896x1344", "2:3 · FHD · 1088x1632",
    "21:9 · SD · 1344x576", "21:9 · HD · 1680x720", "21:9 · FHD · 2016x864"]

SAMPLERS = ["euler", "euler_cfg_pp", "euler_ancestral", "euler_ancestral_cfg_pp", "heun", "heunpp2",
    "dpm_2", "dpm_2_ancestral", "lms", "dpm_fast", "dpm_adaptive", "dpmpp_2s_ancestral",
    "dpmpp_2s_ancestral_cfg_pp", "dpmpp_sde", "dpmpp_sde_gpu", "dpmpp_2m", "dpmpp_2m_cfg_pp",
    "dpmpp_2m_sde", "dpmpp_2m_sde_gpu", "dpmpp_3m_sde", "dpmpp_3m_sde_gpu", "ddpm", "lcm",
    "ipndm", "ipndm_v", "deis", "res_multistep", "res_multistep_cfg_pp", "res_multistep_ancestral",
    "res_multistep_ancestral_cfg_pp", "gradient_estimation", "gradient_estimation_cfg_pp", "er_sde",
    "seeds_2", "seeds_3", "sa_solver", "sa_solver_pece", "ddim", "uni_pc", "uni_pc_bh2"]
SCHEDULERS = ["normal", "karras", "exponential", "sgm_uniform", "simple", "ddim_uniform", "beta",
    "linear_quadratic", "kl_optimal"]

NONE_CHOICE = "— none —"

# ============================================================
# HELPERS
# ============================================================
def _model_dir(family):
    return os.path.join(WORKSPACE, "models/checkpoints" if FAMILIES[family]["loader"] == "checkpoint" else "models/unet")

def _list_files(folder):
    if not os.path.isdir(folder):
        return []
    return sorted(f for f in os.listdir(folder)
                  if not f.endswith(".aria2") and os.path.isfile(os.path.join(folder, f)))

def list_models(family):
    return _list_files(_model_dir(family))

def list_loras():
    return _list_files(os.path.join(WORKSPACE, "models/loras"))

def _resolve_resolution(resolution, custom_w, custom_h):
    if resolution != "Custom":
        dims = resolution.split("·")[-1].strip()
        w, h = (int(v) for v in dims.split("x"))
        return w, h
    w, h = max(64, int(custom_w)), max(64, int(custom_h))
    return (w // 8) * 8, (h // 8) * 8

def _api(path, payload=None):
    data = json.dumps(payload).encode("utf-8") if payload is not None else None
    req = urllib.request.Request(COMFY_URL + path, data=data)
    return urllib.request.urlopen(req)

def build_workflow(family, model_file, loras, prompt, negative, width, height, batch,
                   steps, cfg, sampler, scheduler, shift, guidance, seed):
    FAM = FAMILIES[family]
    wf = {}
    if FAM["loader"] == "checkpoint":
        wf["10"] = {"inputs": {"ckpt_name": model_file}, "class_type": "CheckpointLoaderSimple"}
        model_link, clip_link, vae_link = ["10", 0], ["10", 1], ["10", 2]
    else:
        if model_file.lower().endswith(".gguf"):
            wf["10"] = {"inputs": {"unet_name": model_file}, "class_type": "UnetLoaderGGUF"}
        else:
            wf["10"] = {"inputs": {"unet_name": model_file, "weight_dtype": "default"}, "class_type": "UNETLoader"}
        model_link = ["10", 0]
        c = FAM["clip"]
        if c[0] == "CLIPLoader":
            wf["11"] = {"inputs": {"clip_name": c[1], "type": c[2], "device": "default"}, "class_type": "CLIPLoader"}
        elif c[0] == "DualCLIPLoader":
            wf["11"] = {"inputs": {"clip_name1": c[1], "clip_name2": c[2], "type": c[3], "device": "default"}, "class_type": "DualCLIPLoader"}
        else:
            wf["11"] = {"inputs": {"clip_name1": c[1], "clip_name2": c[2], "clip_name3": c[3], "clip_name4": c[4]}, "class_type": "QuadrupleCLIPLoader"}
        clip_link = ["11", 0]
        wf["12"] = {"inputs": {"vae_name": FAM["vae"]}, "class_type": "VAELoader"}
        vae_link = ["12", 0]

    for idx, (lora_name, lora_strength) in enumerate(loras):
        node_id = str(20 + idx)
        if FAM["loader"] == "checkpoint":
            wf[node_id] = {"inputs": {"lora_name": lora_name, "strength_model": lora_strength,
                                      "strength_clip": lora_strength, "model": model_link, "clip": clip_link},
                           "class_type": "LoraLoader"}
            model_link, clip_link = [node_id, 0], [node_id, 1]
        else:
            wf[node_id] = {"inputs": {"lora_name": lora_name, "strength_model": lora_strength, "model": model_link},
                           "class_type": "LoraLoaderModelOnly"}
            model_link = [node_id, 0]

    if FAM["shift"]:
        wf["30"] = {"inputs": {"shift": shift, "model": model_link}, "class_type": FAM["shift"][0]}
        model_link = ["30", 0]

    wf["31"] = {"inputs": {"text": prompt, "clip": clip_link}, "class_type": "CLIPTextEncode"}
    positive_link = ["31", 0]
    wf["32"] = {"inputs": {"text": negative, "clip": clip_link}, "class_type": "CLIPTextEncode"}
    negative_link = ["32", 0]
    if FAM["guidance"] is not None:
        wf["33"] = {"inputs": {"guidance": guidance, "conditioning": positive_link}, "class_type": "FluxGuidance"}
        positive_link = ["33", 0]

    wf["40"] = {"inputs": {"width": width, "height": height, "batch_size": batch}, "class_type": FAM["latent"]}

    if FAM["sampling"] == "flux2":
        wf["50"] = {"inputs": {"noise_seed": seed}, "class_type": "RandomNoise"}
        wf["51"] = {"inputs": {"model": model_link, "conditioning": positive_link}, "class_type": "BasicGuider"}
        wf["52"] = {"inputs": {"sampler_name": sampler}, "class_type": "KSamplerSelect"}
        wf["53"] = {"inputs": {"steps": steps, "width": width, "height": height}, "class_type": "Flux2Scheduler"}
        wf["54"] = {"inputs": {"noise": ["50", 0], "guider": ["51", 0], "sampler": ["52", 0],
                               "sigmas": ["53", 0], "latent_image": ["40", 0]},
                    "class_type": "SamplerCustomAdvanced"}
    else:
        wf["54"] = {"inputs": {"seed": seed, "steps": steps, "cfg": cfg, "sampler_name": sampler,
                               "scheduler": scheduler, "denoise": 1, "model": model_link,
                               "positive": positive_link, "negative": negative_link, "latent_image": ["40", 0]},
                    "class_type": "KSampler"}
    wf["60"] = {"inputs": {"samples": ["54", 0], "vae": vae_link}, "class_type": "VAEDecode"}
    wf["61"] = {"inputs": {"filename_prefix": "studio", "images": ["60", 0]}, "class_type": "SaveImage"}
    return wf

def start_comfy_server():
    try:
        urllib.request.urlopen(COMFY_URL)
        print("🟢 ComfyUI server already running.")
    except Exception:
        print("🚀 Starting ComfyUI server...")
        subprocess.Popen([sys.executable, "main.py"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        while True:
            try:
                urllib.request.urlopen(COMFY_URL)
                print("🟢 ComfyUI server is up.")
                break
            except Exception:
                time.sleep(2)

# === UI ===  (everything below requires gradio)
try:
    import gradio as gr
except ImportError:
    print("📦 Installing Gradio...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gradio"], check=True)
    import gradio as gr

start_comfy_server()

def generate(family, model_file, prompt, negative, resolution, custom_w, custom_h, batch,
             steps, cfg, sampler, scheduler, shift, guidance, seed,
             l1, s1, l2, s2, l3, s3, l4, s4, l5, s5, l6, s6,
             progress=gr.Progress()):
    if not model_file or model_file == NONE_CHOICE:
        raise gr.Error("No model selected. Download one in Colab Cell 2, then hit 🔄 Refresh files.")
    if not prompt.strip():
        raise gr.Error("Prompt is empty — describe what you want to generate.")

    loras = [(n, s) for n, s in [(l1, s1), (l2, s2), (l3, s3), (l4, s4), (l5, s5), (l6, s6)]
             if n and n != NONE_CHOICE and s != 0]
    used_seed = int(seed) if int(seed) > 0 else random.randint(1, 1125899906842624)
    width, height = _resolve_resolution(resolution, custom_w, custom_h)

    wf = build_workflow(family, model_file, loras, prompt, negative, width, height, int(batch),
                        int(steps), float(cfg), sampler, scheduler, float(shift), float(guidance), used_seed)
    t0 = time.time()
    try:
        res = _api("/prompt", {"prompt": wf})
        prompt_id = json.loads(res.read())["prompt_id"]
    except urllib.error.HTTPError as e:
        body = e.read().decode("utf-8", errors="replace")
        try:
            msg = json.loads(body).get("error", {}).get("message", body[:400])
        except Exception:
            msg = body[:400]
        raise gr.Error(f"ComfyUI rejected the workflow: {msg}")

    progress(0.05, desc="Queued...")
    outputs = None
    while True:
        try:
            hist = json.loads(_api(f"/history/{prompt_id}").read())
            if prompt_id in hist:
                outputs = hist[prompt_id]["outputs"]
                break
        except Exception:
            pass
        elapsed = time.time() - t0
        progress(min(0.05 + elapsed / 120.0, 0.95), desc=f"Sampling... {elapsed:.0f}s")
        time.sleep(1)

    images = []
    for node_output in outputs.values():
        for image in node_output.get("images", []):
            sub = image.get("subfolder", "")
            images.append(os.path.join(WORKSPACE, "output", sub, image["filename"]))
    took = time.time() - t0

    FAM = FAMILIES[family]
    lora_txt = ", ".join(f"{n} @{s}" for n, s in loras) if loras else "none"
    info = (f"✅ {len(images)} image(s) in {took:.1f}s\n"
            f"Family: {family} | Model: {model_file}\n"
            f"Size: {width}x{height} | Steps: {steps} | Sampler: {sampler}/{scheduler}")
    if FAM["sampling"] != "flux2":
        info += f" | CFG: {cfg}"
    if FAM["shift"]:
        info += f" | Shift: {shift}"
    if FAM["guidance"] is not None:
        info += f" | Guidance: {guidance}"
    info += f"\nSeed: {used_seed}\nLoRAs: {lora_txt}"
    return images, info, str(used_seed)

def on_family_change(family):
    FAM = FAMILIES[family]
    d = FAM["defaults"]
    models = list_models(family)
    shift_val = FAM["shift"][1] if FAM["shift"] else 3.0
    guidance_val = FAM["guidance"] if FAM["guidance"] is not None else 3.5
    return (gr.update(choices=models, value=models[0] if models else None),
            gr.update(value=d["steps"]), gr.update(value=d["cfg"], interactive=FAM["sampling"] != "flux2"),
            gr.update(value=d["sampler"]), gr.update(value=d["scheduler"], interactive=FAM["sampling"] != "flux2"),
            gr.update(value=shift_val, interactive=FAM["shift"] is not None),
            gr.update(value=guidance_val, interactive=FAM["guidance"] is not None),
            gr.update(value=f"💡 **{family}** — {FAM['tip']}"))

def refresh_files(family):
    models = list_models(family)
    loras = [NONE_CHOICE] + list_loras()
    model_update = gr.update(choices=models, value=models[0] if models else None)
    lora_updates = [gr.update(choices=loras) for _ in range(6)]
    return [model_update] + lora_updates

def load_session_gallery():
    out = os.path.join(WORKSPACE, "output")
    if not os.path.isdir(out):
        return []
    files = []
    for root, _, names in os.walk(out):
        for n in names:
            if n.lower().endswith((".png", ".jpg", ".jpeg", ".webp")):
                files.append(os.path.join(root, n))
    return sorted(files, key=os.path.getmtime, reverse=True)

def free_vram():
    try:
        _api("/free", {"unload_models": True, "free_memory": True})
        return "🧹 Models unloaded, VRAM freed. Next generation will reload the model."
    except Exception as e:
        return f"⚠️ Could not free VRAM: {e}"

def interrupt_gen():
    try:
        _api("/interrupt", {})
        return "⏹ Interrupt sent — current generation is stopping."
    except Exception as e:
        return f"⚠️ Could not interrupt: {e}"

_css = """
.gradio-container {max-width: 1200px !important; margin: auto;}
footer {display: none !important;}

/* --- Gallery fullscreen/preview fixes --- */
/* Reposition the top-right control buttons so they aren't misplaced/cut off
   when an image is enlarged, and give them breathing room. */
.gradio-container .grid-wrap .icon-buttons,
.gradio-container .preview .icon-buttons,
.gradio-container div[class*="icon-buttons"] {
    right: 16px !important;
    top: 12px !important;
    gap: 10px !important;
    display: flex !important;
    z-index: 60 !important;
}
.gradio-container div[class*="icon-buttons"] button,
.gradio-container .preview button[aria-label] {
    background: rgba(0,0,0,0.55) !important;
    border-radius: 8px !important;
    padding: 4px !important;
}
/* Make the close/fullscreen button always visible and clickable */
.gradio-container button[aria-label="Close"],
.gradio-container button[aria-label="Fullscreen"],
.gradio-container button[aria-label="Exit fullscreen"] {
    opacity: 1 !important;
    pointer-events: auto !important;
}
/* Status pill for the busy indicator */
#studio_status {font-family: monospace; font-size: 13px; padding: 4px 0;}
"""

# Just the status pill markup. All JS lives in event handlers (js=) and demo.load,
# which Gradio reliably executes — gr.HTML <script> tags are often stripped.
_signal_html = '<div id="studio_status">🟢 Idle — ready to generate.</div>'

# Runs once on app load: request notification permission + install the gallery
# close-(x) fix (sends Escape so Gradio fully exits the enlarged preview state).
_init_js = """
() => {
  try { if (window.Notification && Notification.permission === 'default') Notification.requestPermission(); } catch(e){}
  if (!window.__studioClose) {
    window.__studioClose = true;
    document.addEventListener('click', function(ev){
      const btn = ev.target.closest && ev.target.closest('button[aria-label="Close"], button[title="Close"]');
      if (btn) {
        setTimeout(function(){
          try { document.dispatchEvent(new KeyboardEvent('keydown', {key:'Escape', keyCode:27, which:27, bubbles:true})); } catch(e){}
        }, 30);
      }
    }, true);
  }
}
"""

_fam0 = FAMILIES[DEFAULT_FAMILY]
_models0 = list_models(DEFAULT_FAMILY)
_loras0 = [NONE_CHOICE] + list_loras()

with gr.Blocks(css=_css, title="Z-Image Turbo Studio", theme=gr.themes.Soft(primary_hue="emerald", neutral_hue="slate")) as demo:
    gr.Markdown("# ⚡ Z-Image Turbo Studio\n*Powered by your Colab GPU — models load on first generation (first run per model is slower).*")
    with gr.Row():
        with gr.Column(scale=5):
            family = gr.Dropdown(list(FAMILIES.keys()), value=DEFAULT_FAMILY, label="🧬 Model Family")
            model_file = gr.Dropdown(_models0, value=_models0[0] if _models0 else None, label="🧠 Model")
            tip_md = gr.Markdown(f"💡 **{DEFAULT_FAMILY}** — {_fam0['tip']}")
            prompt = gr.Textbox(label="Prompt", lines=3, placeholder="Describe your image...")
            negative = gr.Textbox(label="Negative Prompt", lines=2, value="blurry, low quality, deformed, artifacts")
            with gr.Row():
                resolution = gr.Dropdown(RESOLUTIONS, value="16:9 · FHD · 1920x1080", label="📐 Resolution")
                batch = gr.Slider(1, 4, value=1, step=1, label="Batch")
            with gr.Row():
                custom_w = gr.Number(value=1024, label="Custom W (Custom only)", precision=0)
                custom_h = gr.Number(value=1024, label="Custom H (Custom only)", precision=0)
            with gr.Accordion("🎨 LoRA Stack (up to 6 — strength 0 disables a slot)", open=True):
                lora_dd, lora_sl = [], []
                for i in range(6):
                    with gr.Row():
                        lora_dd.append(gr.Dropdown(_loras0, value=NONE_CHOICE, label=f"LoRA {i+1}", scale=3))
                        lora_sl.append(gr.Slider(0.0, 2.0, value=1.0, step=0.05, label="Strength", scale=2))
            with gr.Accordion("⚙️ Sampling (auto-filled per family — tweak freely)", open=False):
                with gr.Row():
                    steps = gr.Slider(1, 60, value=_fam0["defaults"]["steps"], step=1, label="Steps")
                    cfg = gr.Number(value=_fam0["defaults"]["cfg"], label="CFG")
                with gr.Row():
                    sampler = gr.Dropdown(SAMPLERS, value=_fam0["defaults"]["sampler"], label="Sampler")
                    scheduler = gr.Dropdown(SCHEDULERS, value=_fam0["defaults"]["scheduler"], label="Scheduler")
                with gr.Row():
                    shift = gr.Slider(0.5, 10.0, value=_fam0["shift"][1] if _fam0["shift"] else 3.0, step=0.1,
                                      label="Shift", interactive=_fam0["shift"] is not None)
                    guidance = gr.Number(value=_fam0["guidance"] if _fam0["guidance"] is not None else 3.5,
                                         label="Flux Guidance", interactive=_fam0["guidance"] is not None)
                with gr.Row():
                    seed = gr.Number(value=0, label="Seed (0 = random)", precision=0)
                    last_seed = gr.Textbox(label="Last seed", interactive=False)
                    reuse_btn = gr.Button("♻️ Reuse last seed", size="sm")
            with gr.Row():
                generate_btn = gr.Button("✨ Generate", variant="primary", scale=3)
                interrupt_btn = gr.Button("⏹ Stop", scale=1)
            with gr.Row():
                refresh_btn = gr.Button("🔄 Refresh files", size="sm")
                vram_btn = gr.Button("🧹 Free VRAM", size="sm")
        with gr.Column(scale=6):
            gr.HTML(_signal_html)
            gallery = gr.Gallery(label="Results", columns=2, height=560, preview=True)
            info_box = gr.Textbox(label="Generation info", lines=6, interactive=False)
            with gr.Accordion("🖼 Session gallery (everything generated this session)", open=False):
                session_btn = gr.Button("Load session gallery", size="sm")
                session_gallery = gr.Gallery(columns=4, height=420)

    family.change(on_family_change, inputs=[family],
                  outputs=[model_file, steps, cfg, sampler, scheduler, shift, guidance, tip_md])
    refresh_btn.click(refresh_files, inputs=[family], outputs=[model_file] + lora_dd)
    reuse_btn.click(lambda s: gr.update(value=int(s)) if s.strip().isdigit() else gr.update(),
                    inputs=[last_seed], outputs=[seed])
    vram_btn.click(free_vram, outputs=[info_box])
    interrupt_btn.click(interrupt_gen, outputs=[info_box])
    session_btn.click(load_session_gallery, outputs=[session_gallery])

    gen_inputs = [family, model_file, prompt, negative, resolution, custom_w, custom_h, batch,
                  steps, cfg, sampler, scheduler, shift, guidance, seed]
    for i in range(6):
        gen_inputs.extend([lora_dd[i], lora_sl[i]])

    # Busy indicator: an INDEPENDENT client-side listener — self-contained so it
    # never depends on <script> execution and never blocks generation.
    _busy_js = ("() => { const s=document.getElementById('studio_status');"
                " if(s){s.textContent='⏳ Generating on your Colab GPU…';s.style.color='#e0a83b';}"
                " try{document.title='⏳ Generating… · Image Studio';}catch(e){} }")
    generate_btn.click(None, None, None, js=_busy_js)

    # PRIMARY generation event (this exact standalone call is the proven-working one).
    _gen_event = generate_btn.click(generate, inputs=gen_inputs, outputs=[gallery, info_box, last_seed])

    # Completion notify + sound: a .then() off the REAL backend event, fully self-contained JS.
    _done_js = ("() => { const s=document.getElementById('studio_status');"
                " if(s){s.textContent='✅ Done — image ready';s.style.color='#2ecc71';}"
                " try{document.title='✅ Done · Image Studio';}catch(e){}"
                " try{const C=window.AudioContext||window.webkitAudioContext;"
                " if(C){const c=new C(),o=c.createOscillator(),g=c.createGain();"
                " o.type='sine';o.frequency.value=880;o.connect(g);g.connect(c.destination);"
                " g.gain.setValueAtTime(0.0001,c.currentTime);"
                " g.gain.exponentialRampToValueAtTime(0.25,c.currentTime+0.02);"
                " g.gain.exponentialRampToValueAtTime(0.0001,c.currentTime+0.35);"
                " o.start();o.stop(c.currentTime+0.36);}}catch(e){}"
                " try{if(window.Notification&&Notification.permission==='granted'){"
                " const n=new Notification('🎨 Image Studio',{body:'Your generation is ready!'});"
                " setTimeout(()=>{try{n.close();}catch(e){}},6000);}"
                " else if(window.Notification&&Notification.permission==='default'){Notification.requestPermission();}}catch(e){} }")
    _gen_event.then(None, None, None, js=_done_js)

    # Run init JS on load (permission request + gallery close-fix).
    demo.load(None, None, None, js=_init_js)

auth = ("studio", GRADIO_PASSWORD.strip()) if GRADIO_PASSWORD.strip() else None
print("\n" + "=" * 60)
print("🌐 LAUNCHING WEB STUDIO — your public link appears below.")
print("   Keep this Colab tab OPEN while you work in the link!")
if auth:
    print(f"   Login → user: studio | password: (the one you set)")
print("=" * 60 + "\n")
demo.queue().launch(share=True, auth=auth, show_error=True)


In [ ]:
#@title 4. 🔇 Keep-Alive (optional — prevents idle disconnect)
#@markdown Run this, then **click ▶ Play on the audio player below once**. It loops a silent audio track, which keeps this Colab tab registered as "active" so Colab's ~90-minute **idle** disconnect doesn't kick in while you're working in the Studio tab.
#@markdown
#@markdown **Honest notes:** the Colab tab must stay open (minimized/background is fine) · your browser may require that one manual click on Play (autoplay-with-sound is blocked until you interact) · this does **not** extend Colab's hard session caps — it only prevents the idle timeout.

from IPython.display import display, HTML

_SILENT_WAV_B64 = "..."

display(HTML(
    '<div style="background:#161b22;border:1px solid #30363d;border-radius:10px;'
    'padding:12px 16px;font-family:monospace;max-width:520px;">'
    '<div style="color:#2ecc71;font-weight:bold;">🔇 Keep-Alive active</div>'
    '<div style="color:#8b949e;font-size:12px;margin:4px 0 8px 0;">'
    'Press ▶ once. Leave this tab open (background is fine) and work in the Studio tab.</div>'
    '<audio controls loop style="width:100%;" '
    'src="data:audio/wav;base64,' + _SILENT_WAV_B64 + '"></audio>'
    '</div>'))

print("✅ Keep-alive player ready — press Play once and you're covered.")


In [ ]:
 #@title 4. Export & Download Results
#@markdown Run this to instantly zip and download all the generated images from this session.

import os
from google.colab import files

OUTPUT_DIR = "/content/ComfyUI/output"
ZIP_NAME = "/content/Z_Image_Artworks.zip"

if os.path.exists(OUTPUT_DIR) and len(os.listdir(OUTPUT_DIR)) > 0:
    print("🗜️ Zipping generated artworks...")
    !zip -j -q {ZIP_NAME} {OUTPUT_DIR}/*.png
    print("📥 Initiating download...")
    files.download(ZIP_NAME)
else:
    print("⚠️ No images found in the output directory yet!")